# How to Fine-Tune an LLM: Making a small Ollama model for task-specific knowledge

Welcome to Local AI fine-tuning. In this tutorial, we will take a base LLM (Llama-3-8B), fine-tune it on a custom dataset, and export it as a `GGUF` file. This file can then be loaded seamlessly into Ollama.

### The Analytical Pipeline:
1. **Environment Setup:** Importing Unsloth and PyTorch.
2. **Model Quantization:** Loading the base model in 4-bit precision to fit into consumer GPU memory.
3. **LoRA Injection:** Applying Low-Rank Adapters to the model's attention layers.
4. **Data Formatting:** Structuring our custom knowledge into a supervised instruction format.
5. **Training (SFT):** Running the Supervised Fine-Tuning Trainer.
6. **Ollama Export:** Merging the LoRA weights and converting to GGUF.

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, you must install Unsloth and its dependencies:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps trl peft accelerate bitsandbytes

import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Set maximum sequence length (context window) for training
# Higher context uses exponentially more VRAM. 2048 is a safe analytical baseline.
max_seq_length = 2048 

print("Libraries imported successfully. Ready to fine-tune!")

## Step 1: Loading the Base Model with Quantization

An 8-Billion parameter model requires roughly 16GB of VRAM just to load at standard 16-bit float precision. To train it, you need even more.

We will use **4-bit Quantization**. This mathematically compresses the pre-trained weights, reducing the memory footprint by 4x with negligible loss in reasoning accuracy. Unsloth handles this via `bitsandbytes` under the hood.

In [ ]:
# Cell 4: Loading the Model and Tokenizer

# We use the unsloth pre-quantized Llama-3 model for speed
model_name = "unsloth/llama-3-8b-bnb-4bit"

print(f"Loading {model_name} in 4-bit precision...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None, # Auto-detects fp16 or bf16 based on hardware
    load_in_4bit=True, # Mathematically compress weights
)

print("Model and Tokenizer successfully loaded into VRAM.")

## Step 2: Injecting LoRA Adapters

The base model is currently loaded, but its weights are frozen. We now inject the **LoRA matrices** ($A$ and $B$). 

We target the specific weight matrices inside the Transformer's Attention mechanism (e.g., Query, Key, Value, Output projections). 

**Analytical Hyperparameters:**
* `r` (Rank): The dimension of the inner matrices. Higher rank means more learning capacity, but more compute. 16 is standard.
* `lora_alpha`: A scaling factor that determines how heavily the LoRA weights influence the base model. Typically set to $2 \times r$.

In [ ]:
# Cell 6: Applying PEFT / LoRA

model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0, # 0 is optimized for speed
    bias="none",    # Optimized for speed
    use_gradient_checkpointing="unsloth", # Saves massive amounts of VRAM
    random_state=3407,
    use_rslora=False,
)

print("LoRA adapters successfully injected. Model is ready for supervised learning.")
# Analytically, we have reduced trainable parameters from ~8,000,000,000 to roughly ~40,000,000.

## Step 3: Formatting the Dataset

To teach an LLM a specific task, we use **Supervised Fine-Tuning (SFT)**. We must provide data in an Instruction-Response format. 

We will define a prompt template and map our dataset to it. The model will look at the `Instruction` and `Input`, and backpropagate gradients to mathematically minimize the difference between its own output and the provided `Response`.

In [ ]:
# Cell 8: Dataset Preparation

# Standard Alpaca Instruction Prompt format
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# EOS (End of Sequence) token is strictly required. 
# Without it, the model will never learn when to stop generating text!
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    
    # Iterate through the batch and format
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
        
    return { "text" : texts }

# Load a sample dataset (e.g., a dataset teaching the model Python code)
print("Loading dataset...")
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# Map the formatting function to the dataset
print("Applying prompt template mapping...")
formatted_dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset ready. Formatted {len(formatted_dataset)} samples.")

## Step 4: The Fine-Tuning Loop

We use the `SFTTrainer` from the Hugging Face `trl` (Transformer Reinforcement Learning) library. 

It handles the complex PyTorch training loop under the hood. We define our `TrainingArguments` to control the learning rate, batch size, and weight decay optimization.

In [ ]:
# Cell 10: Executing the Training Run

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Can make training 5x faster for short sequences
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4, # Simulates a larger batch size of 8
        warmup_steps=5,
        max_steps=60, # For a full run, use num_train_epochs=1 instead of max_steps
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit", # Use 8-bit Adam optimizer to save VRAM
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("--- Starting Backpropagation ---")
trainer_stats = trainer.train()
print("--- Training Complete ---")

# Print analytical metrics
print(f"Total training time: {trainer_stats.metrics['train_runtime']} seconds")
print(f"Final training loss: {trainer_stats.metrics['train_loss']}")

## Step 5: Exporting to Ollama (GGUF)

We have successfully trained the LoRA matrices. However, Ollama uses `C++` (specifically `llama.cpp`) to run models blazing fast on local CPUs and GPUs. It requires models to be in the **GGUF** format.

Unsloth provides a massive convenience function here. `save_pretrained_gguf` mathematically merges our trained LoRA matrices ($BA$) back into the base weights ($W_0$), quantizes the whole thing to a lightweight format (like `q4_k_m`), and writes the `.gguf` file required by Ollama!

In [ ]:
# Cell 12: GGUF Export for Ollama

export_path = "my_custom_model"

print("Merging LoRA adapters into base model and converting to GGUF format...")
print("This requires heavy matrix multiplication and may take several minutes.")

# Export to 4-bit GGUF (Highly recommended for Ollama to balance speed and accuracy)
model.save_pretrained_gguf(export_path, tokenizer, quantization_method="q4_k_m")

print("\n--- Export Successful! ---")
print(f"Your model is now saved in the '{export_path}' directory.")
print("\nTo use it in Ollama, create a file named 'Modelfile' with this text:")
print(f"FROM ./unsloth.Q4_K_M.gguf")
print("Then run: ollama create my-fine-tuned-model -f Modelfile")
print("Finally run: ollama run my-fine-tuned-model")